In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from imblearn.over_sampling import SMOTE

import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")

In [3]:
df = pd.read_csv('transaction.csv')
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset shape: (9841, 51)
Rows: 9841
Columns: 51


In [5]:
print(df.head())
print(df.dtypes)
print(df.info())

   Unnamed: 0  Index                                     Address  FLAG  \
0           0      1  0x00009277775ac7d0d59eaad8fee3d10ac6c805e8     0   
1           1      2  0x0002b44ddb1476db43c868bd494422ee4c136fed     0   
2           2      3  0x0002bda54cb772d040f779e88eb453cac0daa244     0   
3           3      4  0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e     0   
4           4      5  0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89     0   

   Avg min between sent tnx  Avg min between received tnx  \
0                    844.26                       1093.71   
1                  12709.07                       2958.44   
2                 246194.54                       2434.02   
3                  10219.60                      15785.09   
4                     36.61                      10707.77   

   Time Diff between first and last (Mins)  Sent tnx  Received Tnx  \
0                                704785.63       721            89   
1                               1218216.73      

In [6]:
print("Class balance:")
print(df['FLAG'].value_counts())
print()
print(f"Legitimate wallets: {df['FLAG'].value_counts()[0]}")
print(f"Fraudulent wallets: {df['FLAG'].value_counts()[1]}")
print(f"Fraud percentage: {round(df['FLAG'].value_counts()[1] / len(df) * 100, 2)}%")

Class balance:
FLAG
0    7662
1    2179
Name: count, dtype: int64

Legitimate wallets: 7662
Fraudulent wallets: 2179
Fraud percentage: 22.14%


In [9]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Columns with missing values: {df.isnull().any().sum()}")

Missing values per column:
Unnamed: 0                                              0
Index                                                   0
Address                                                 0
FLAG                                                    0
Avg min between sent tnx                                0
Avg min between received tnx                            0
Time Diff between first and last (Mins)                 0
Sent tnx                                                0
Received Tnx                                            0
Number of Created Contracts                             0
Unique Received From Addresses                          0
Unique Sent To Addresses                                0
min value received                                      0
max value received                                      0
avg val received                                        0
min val sent                                            0
max val sent                                 

In [8]:
# Fill missing values with 0
df = df.fillna(0)

# Confirm no missing values remain
print(f"Missing values after cleaning: {df.isnull().sum().sum()}")

Missing values after cleaning: 0


In [10]:
# Check for duplicates
print(f"Duplicate rows before cleaning: {df.duplicated().sum()}")

# Remove duplicates
df = df.drop_duplicates()

# Confirm
print(f"Duplicate rows after cleaning: {df.duplicated().sum()}")
print(f"Dataset shape after removing duplicates: {df.shape}")

Duplicate rows before cleaning: 0
Duplicate rows after cleaning: 0
Dataset shape after removing duplicates: (9841, 51)


In [ ]:

df = df.drop(columns=['Address', 'Unnamed: 0', 'Index'])


print(f"Dataset shape after dropping irrelevant columns: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Dataset shape after dropping irrelevant columns: (9841, 48)
Remaining columns: ['FLAG', 'Avg min between sent tnx', 'Avg min between received tnx', 'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx', 'Number of Created Contracts', 'Unique Received From Addresses', 'Unique Sent To Addresses', 'min value received', 'max value received ', 'avg val received', 'min val sent', 'max val sent', 'avg val sent', 'min value sent to contract', 'max val sent to contract', 'avg value sent to contract', 'total transactions (including tnx to create contract', 'total Ether sent', 'total ether received', 'total ether sent contracts', 'total ether balance', ' Total ERC20 tnxs', ' ERC20 total Ether received', ' ERC20 total ether sent', ' ERC20 total Ether sent contract', ' ERC20 uniq sent addr', ' ERC20 uniq rec addr', ' ERC20 uniq sent addr.1', ' ERC20 uniq rec contract addr', ' ERC20 avg time between sent tnx', ' ERC20 avg time between rec tnx', ' ERC20 avg time between rec 2 tnx', ' 

In [ ]:

X = df.drop(columns=['FLAG'])
y = df['FLAG']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {X.shape[1]}")

Features shape: (9841, 47)
Target shape: (9841,)
Feature columns: 47


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"Training fraud cases: {y_train.sum()}")
print(f"Testing fraud cases: {y_test.sum()}")

Training set size: (7872, 47)
Testing set size: (1969, 47)
Training fraud cases: 1743
Testing fraud cases: 436


In [ ]:

print(df.dtypes[df.dtypes == 'object'])

ERC20 most sent token type    object
ERC20_most_rec_token_type     object
dtype: object


In [ ]:

X_train = X_train.drop(columns=[' ERC20 most sent token type', ' ERC20_most_rec_token_type'], errors='ignore')
X_test = X_test.drop(columns=[' ERC20 most sent token type', ' ERC20_most_rec_token_type'], errors='ignore')

print(f"X_train shape after dropping text columns: {X_train.shape}")
print(f"X_test shape after dropping text columns: {X_test.shape}")


smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE - Training set size: {X_train.shape}")
print(f"After SMOTE - Training set size: {X_train_smote.shape}")
print(f"Before SMOTE - Fraud cases: {y_train.sum()}")
print(f"After SMOTE - Fraud cases: {y_train_smote.sum()}")
print(f"After SMOTE - Legitimate cases: {(y_train_smote == 0).sum()}")

X_train shape after dropping text columns: (7872, 45)
X_test shape after dropping text columns: (1969, 45)
Before SMOTE - Training set size: (7872, 45)
After SMOTE - Training set size: (12258, 45)
Before SMOTE - Fraud cases: 1743
After SMOTE - Fraud cases: 6129
After SMOTE - Legitimate cases: 6129


In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

print(f"Training set scaled shape: {X_train_scaled.shape}")
print(f"Testing set scaled shape: {X_test_scaled.shape}")
print(f"Feature mean after scaling: {X_train_scaled.mean().round(4)}")
print(f"Feature std after scaling: {X_train_scaled.std().round(4)}")

Training set scaled shape: (12258, 45)
Testing set scaled shape: (1969, 45)
Feature mean after scaling: 0.0
Feature std after scaling: 0.9189


In [ ]:

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train_smote)


lr_pred = lr_model.predict(X_test_scaled)


print("=== Logistic Regression Results ===")
print(classification_report(y_test, lr_pred))
print(f"ROC-AUC Score: {round(roc_auc_score(y_test, lr_pred), 4)}")

=== Logistic Regression Results ===
              precision    recall  f1-score   support

           0       0.93      0.54      0.69      1533
           1       0.35      0.85      0.49       436

    accuracy                           0.61      1969
   macro avg       0.64      0.70      0.59      1969
weighted avg       0.80      0.61      0.64      1969

ROC-AUC Score: 0.6971


In [ ]:

rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train_scaled, y_train_smote)


rf_pred = rf_model.predict(X_test_scaled)


print("=== Random Forest Results ===")
print(classification_report(y_test, rf_pred))
print(f"ROC-AUC Score: {round(roc_auc_score(y_test, rf_pred), 4)}")

=== Random Forest Results ===
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1533
           1       0.94      0.87      0.91       436

    accuracy                           0.96      1969
   macro avg       0.95      0.93      0.94      1969
weighted avg       0.96      0.96      0.96      1969

ROC-AUC Score: 0.9291


In [23]:

xgb_model = xgb.XGBClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train_scaled, y_train_smote)


xgb_pred = xgb_model.predict(X_test_scaled)


print("=== XGBoost Results ===")
print(classification_report(y_test, xgb_pred))
print(f"ROC-AUC Score: {round(roc_auc_score(y_test, xgb_pred), 4)}")

=== XGBoost Results ===
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1533
           1       0.94      0.89      0.92       436

    accuracy                           0.96      1969
   macro avg       0.96      0.94      0.95      1969
weighted avg       0.96      0.96      0.96      1969

ROC-AUC Score: 0.9391


In [24]:
# Model comparison summary
print("=== MODEL COMPARISON SUMMARY ===")
print(f"{'Model':<25} {'Accuracy':<12} {'Fraud Recall':<15} {'Fraud Precision':<18} {'Fraud F1':<12} {'ROC-AUC'}")
print("-" * 90)
print(f"{'Logistic Regression':<25} {'0.61':<12} {'0.85':<15} {'0.35':<18} {'0.49':<12} {'0.6971'}")
print(f"{'Random Forest':<25} {'0.96':<12} {'0.87':<15} {'0.94':<18} {'0.91':<12} {'0.9291'}")
print(f"{'XGBoost (chosen)':<25} {'0.96':<12} {'0.89':<15} {'0.94':<18} {'0.92':<12} {'0.9391'}")
print("-" * 90)
print("Winner: XGBoost — highest fraud recall, F1 and ROC-AUC")

=== MODEL COMPARISON SUMMARY ===
Model                     Accuracy     Fraud Recall    Fraud Precision    Fraud F1     ROC-AUC
------------------------------------------------------------------------------------------
Logistic Regression       0.61         0.85            0.35               0.49         0.6971
Random Forest             0.96         0.87            0.94               0.91         0.9291
XGBoost (chosen)          0.96         0.89            0.94               0.92         0.9391
------------------------------------------------------------------------------------------
Winner: XGBoost — highest fraud recall, F1 and ROC-AUC


In [1]:
# Confusion Matrix for XGBoost
cm = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
plt.title('XGBoost Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"True Negatives (Legit correctly identified): {cm[0][0]}")
print(f"False Positives (Legit wrongly flagged as fraud): {cm[0][1]}")
print(f"False Negatives (Fraud missed): {cm[1][0]}")
print(f"True Positives (Fraud correctly caught): {cm[1][1]}")

NameError: name 'confusion_matrix' is not defined